In [ ]:
# Cell 1: Install the PEFT Stack
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U bitsandbytes
!pip install -q -U trl
!pip install -q -U datasets
!pip install -q -U accelerate

print("✅ PEFT Stack successfully installed!")

In [ ]:
# Cell 2: Authenticate and Load Data
from huggingface_hub import login
from datasets import load_dataset
import os

# Put your Hugging Face Write Token here
hf_token = "Your_Hugging_Face_Write_Token"
login(token=hf_token)

# Replace with the actual Repo ID you used in Week 8
# e.g., "your-username/pii-redactor-training-v1"
REPO_ID = "your-username/pii-redactor-training-v1"

print(f"Pulling {REPO_ID} from the Hub...")
# Load the dataset directly into the Colab GPU memory
dataset = load_dataset(REPO_ID, split="train")

print(f"✅ Successfully loaded {len(dataset)} training rows.")
print("\nSample Row:")
print(dataset[0]['messages'])

In [ ]:
# Cell 3: Load the Tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# We will use Mistral 7B Instruct v0.3
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_auth_token=hf_token)

# Setting up padding (Critical for batch training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fixes weird formatting bugs during training

print("✅ Tokenizer loaded successfully!")

In [ ]:
# Cell 4: Define the 4-bit Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, # Squeezes out even more memory!
    bnb_4bit_quant_type="nf4",      # NormalFloat4 - optimized for neural net weights
    bnb_4bit_compute_dtype=torch.bfloat16 # The math is still done in 16-bit for accuracy
)
print("✅ BitsAndBytes Configuration set!")

In [ ]:
# Cell 5: Load the Base Model
print("Downloading Base Model into 4-bit memory... (This takes a few minutes)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # Automatically puts the model on the GPU
    token=hf_token
)

# Disable caching to save VRAM during training
model.config.use_cache = False

print("✅ Model successfully loaded into 4-bit VRAM!")